In [5]:
from pyspark.sql import SparkSession 
import pyspark.sql.functions as F 
import pyspark.sql.types as T 

In [6]:
spark = SparkSession.builder.appName('sales').getOrCreate() 

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/20 10:07:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/20 10:07:41 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/10/20 10:07:41 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [7]:
sales = spark.read.csv('sales_data.csv' , header=True , inferSchema= True) 

In [26]:
sales.columns

['Order_ID',
 'Order_Date',
 'Customer_ID',
 'Region',
 'City',
 'Segment',
 'Channel',
 'Category',
 'Subcategory',
 'Product_ID',
 'Quantity',
 'Unit_Price_INR',
 'Discount',
 'Sales_INR',
 'Discount_Amount_INR',
 'Net_Sales_INR',
 'Cost_INR',
 'Profit_INR',
 'Shipping_Days',
 'Returned',
 'Customer_Satisfaction',
 'Payment_Method']

In [9]:
Numiric_types =  {'Quantity' : T.IntegerType(), 
                 'Unit_Price_INR': T.DoubleType(),  
                 'Discount': T.DoubleType(), 
                 'Sales_INR': T.DoubleType(),
                 'Discount_Amount_INR': T.DoubleType(),
                 'Net_Sales_INR':T.DoubleType(),
                 'Cost_INR':T.DoubleType(),
                 'Profit_INR':T.DoubleType(),  
                 'Shipping_Days':T.IntegerType(), 
                 'Customer_Satisfaction':T.IntegerType()} 


                 
               
               

In [10]:
from pyspark.sql.types import StringType

for field in sales.schema.fields:
    if isinstance(field.dataType, StringType):
        col_name = field.name
        sales = sales.withColumn(
            col_name,
            F.when(F.trim(F.col(col_name)) == "", None)
             .otherwise(F.trim(F.col(col_name)))
        )
                     

In [11]:
df_city=sales.groupby('city').agg(
    F.count(F.col('Order_ID')).alias('total orders'), 
    F.round(F.sum(F.col('Sales_INR')),2).alias('Brut sales'), 
    F.round(F.sum(F.col('Net_Sales_INR')),2).alias('Net Sales'), 
    F.round(F.sum(F.col('Profit_INR')),2).alias('Profit')  
).orderBy(F.desc(F.col('Profit'))) 

In [12]:
df_city.show() 

+-----------+------------+----------+----------+----------+
|       city|total orders|Brut sales| Net Sales|    Profit|
+-----------+------------+----------+----------+----------+
|  Hyderabad|         884|4929918.08|4381168.68|1652592.32|
|    Chennai|         904|4830015.79|4326329.35|1637690.64|
|     Mumbai|         872| 4720511.4|4242724.25|1606431.17|
|  Bengaluru|         904|4779211.46|4253292.23|1591984.39|
|      Kochi|         869| 4717515.3|4220718.91|1578886.31|
|    Lucknow|         832|4512364.93|4042279.92|1542969.53|
|  Ahmedabad|         838|4464421.52|3966293.46|1504963.06|
|     Jaipur|         803|4430471.21|3975284.17|1495055.92|
|      Surat|         818|4331012.76|3874034.91|1462960.11|
|      Delhi|         805|4322051.44|3841306.04|1450850.26|
|       Pune|         786|4352288.31|3871891.24|1442705.22|
| Chandigarh|         777|4165845.42|3713336.51| 1406898.8|
|     Nagpur|         650|3696041.43|3296999.06|1254226.64|
|     Bhopal|         652|3532879.95|316

In [13]:
df_city.filter(F.col('total orders') > 700).show()  

+----------+------------+----------+----------+----------+
|      city|total orders|Brut sales| Net Sales|    Profit|
+----------+------------+----------+----------+----------+
| Hyderabad|         884|4929918.08|4381168.68|1652592.32|
|   Chennai|         904|4830015.79|4326329.35|1637690.64|
|    Mumbai|         872| 4720511.4|4242724.25|1606431.17|
| Bengaluru|         904|4779211.46|4253292.23|1591984.39|
|     Kochi|         869| 4717515.3|4220718.91|1578886.31|
|   Lucknow|         832|4512364.93|4042279.92|1542969.53|
| Ahmedabad|         838|4464421.52|3966293.46|1504963.06|
|    Jaipur|         803|4430471.21|3975284.17|1495055.92|
|     Surat|         818|4331012.76|3874034.91|1462960.11|
|     Delhi|         805|4322051.44|3841306.04|1450850.26|
|      Pune|         786|4352288.31|3871891.24|1442705.22|
|Chandigarh|         777|4165845.42|3713336.51| 1406898.8|
+----------+------------+----------+----------+----------+



In [14]:
df_city_product= sales.groupby('city').agg(
    F.count_distinct('Product_ID').alias('Num de produit'),
    F.round(F.avg(F.col('Unit_Price_INR')),2).alias('Avg unit price')) 

In [15]:
df_city_product.show() 

+-----------+--------------+--------------+
|       city|Num de produit|Avg unit price|
+-----------+--------------+--------------+
|      Kochi|            99|       1341.73|
|Bhubaneswar|            99|       1368.52|
|      Patna|            98|       1419.97|
|    Lucknow|            98|       1395.72|
|    Chennai|            99|       1308.73|
|     Mumbai|            99|       1361.67|
|  Ahmedabad|            99|       1318.14|
|    Kolkata|            99|       1347.74|
|      Surat|            99|       1332.85|
|       Pune|            99|       1365.25|
|      Delhi|            99|        1341.0|
|     Raipur|            98|       1312.35|
| Chandigarh|            99|       1342.71|
|     Nagpur|            99|       1377.86|
|     Bhopal|            99|       1350.05|
|  Bengaluru|            99|       1332.92|
|  Hyderabad|            99|       1359.43|
|     Indore|            99|       1286.84|
|   Guwahati|            99|       1288.92|
|     Jaipur|            99|    

In [16]:
df_city=df_city.join(df_city_product, how='left' ,on = 'city' )  

In [17]:
df_city.printSchema()   

root
 |-- city: string (nullable = true)
 |-- total orders: long (nullable = false)
 |-- Brut sales: double (nullable = true)
 |-- Net Sales: double (nullable = true)
 |-- Profit: double (nullable = true)
 |-- Num de produit: long (nullable = true)
 |-- Avg unit price: double (nullable = true)



In [18]:
from pathlib import Path 

In [19]:
Base_dir= Path(__file__).resolve().parent() 

NameError: name '__file__' is not defined

In [22]:
BASE_DIR = Path.cwd() 

In [25]:
T.

PosixPath('/Users/hakim/Desktop')